# Tratamiento de los valores perdidos

Cargamos los datos.

En esta ocasión cargamos los datos desde la carpeta /content/sample_data/ de Colab. Es necesario ubicar ahí el archivo `Ejemplo_valores_perdidos.csv`

In [69]:
import os
import pandas as pd
def cargar_datos():
    data = pd.read_csv(r"C:\americo\ia_dema\z-ejercicios_kaggle\Vehicles_elect_2025\data_real\electric_vehicles_spec_2025.csv")
    return data

data = cargar_datos()

# revisar - consultado con copailot

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Para imputación múltiple
from fancyimpute import IterativeImputer



# ——————————————————————————————————————————————————————————
# A) CARGA DE DATOS
# ——————————————————————————————————————————————————————————
def load_data(path: str) -> pd.DataFrame:
    """Carga el CSV y devuelve un DataFrame."""
    return pd.read_csv(path)



# ——————————————————————————————————————————————————————————
# B) DIAGNÓSTICO DE VALORES PERDIDOS
# ——————————————————————————————————————————————————————————
def diagnose_missing(df: pd.DataFrame, plot: bool = True):
    """Imprime shape, nulos por variable y %,
       y opcionalmente dibuja un barplot y heatmap."""
    n, m = df.shape
    print(f"Shape: {n} filas × {m} columnas\n")
    
    # Conteo y porcentaje
    miss_cnt  = df.isnull().sum()
    miss_pct  = miss_cnt / n * 100
    summary   = pd.DataFrame({
        'missing_count': miss_cnt,
        'missing_pct': miss_pct.round(2)
    }).query("missing_count>0")
    display(summary)

    if plot and not summary.empty:
        # Barra de % faltantes
        plt.figure(figsize=(8,4))
        summary['missing_pct'].sort_values().plot.barh()
        plt.title("% valores perdidos por columna")
        plt.tight_layout()
        plt.show()

        # Heatmap de correlación de nulos
        sub = df.loc[:, summary.index].isnull()
        plt.figure(figsize=(6,6))
        sns.heatmap(sub.corr(), annot=True, cmap='coolwarm')
        plt.title("Correlación de patrones de NA")
        plt.show()



# ——————————————————————————————————————————————————————————
# C) ELIMINACIÓN DURA DE COLUMNAS
# ——————————————————————————————————————————————————————————
def drop_cols_by_missing(df: pd.DataFrame, pct_thresh: float=0.30) -> pd.DataFrame:
    """Elimina columnas con > pct_thresh (ej. 0.30=30%) de nulos."""
    to_drop = df.columns[df.isnull().mean() > pct_thresh]
    print("Columnas eliminadas (>%.1f%% NA):" % (pct_thresh*100), list(to_drop))
    return df.drop(columns=to_drop).copy()



# ——————————————————————————————————————————————————————————
# D) LIMPIEZA DE FILAS EN UN SUBCONJUNTO DE COLUMNAS
# ——————————————————————————————————————————————————————————
def drop_rows_subset(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Elimina filas que tengan NA en cualquiera de las cols indicadas."""
    before = len(df)
    df2 = df.dropna(subset=cols).copy()
    print(f"Filas antes: {before}, después: {len(df2)} (subset {cols})")
    return df2



# ——————————————————————————————————————————————————————————
# E) IMPUTACIÓN SIMPLE POR MEDIA (solo numéricas)
# ——————————————————————————————————————————————————————————
def simple_impute_mean(df: pd.DataFrame) -> pd.DataFrame:
    """Rellena NA en variables numéricas con su media."""
    means = df.mean(numeric_only=True)
    df2 = df.fillna(means).copy()
    print("Imputación simple: media aplicada a numericals.")
    return df2



# ——————————————————————————————————————————————————————————
# F) IMPUTACIÓN AVANZADA (IterativeImputer)
# ——————————————————————————————————————————————————————————
def advanced_impute_iterative(df: pd.DataFrame,
                              random_state: int=0) -> pd.DataFrame:
    """
    Separa numéricas y no-numéricas, aplica IterativeImputer
    solo a numéricas y concatena de nuevo.
    """
    # columnas numéricas
    num_cols = df.select_dtypes(include='number').columns
    cat_cols = df.columns.difference(num_cols)
    
    imputer = IterativeImputer(random_state=random_state)
    num_filled = imputer.fit_transform(df[num_cols])
    
    # reconstrucción
    df_num = pd.DataFrame(num_filled, columns=num_cols, index=df.index)
    df_cat = df[cat_cols].copy()
    
    print("Imputación múltiple (IterativeImputer) completada.")
    return pd.concat([df_cat, df_num], axis=1)



# ——————————————————————————————————————————————————————————
# G) GUARDADO FINAL
# ——————————————————————————————————————————————————————————
def save_data(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    print("Datos guardados en:", path)



# ——————————————————————————————————————————————————————————
# FLUJO PRINCIPAL
# ——————————————————————————————————————————————————————————
if __name__ == "__main__":
    # 1) Carga
    data = load_data("electric_vehicles_spec_2025.csv")
    
    # 2) Diagnóstico
    diagnose_missing(data)
    
    # 3) Eliminación dura de columnas con >30% NA
    data2 = drop_cols_by_missing(data, pct_thresh=0.30)
    
    # 4) (Opcional) eliminar filas si quieres limpiar un subset:
    clean_subset = drop_rows_subset(data2, cols=['model','top_speed_kmh','battery_capacity_kWh'])
    
    # 5) Imputación simple
    simple_filled = simple_impute_mean(data2)
    diagnose_missing(simple_filled, plot=False)  # verificar quedan solo nulos en catgs
    
    # 6) Imputación avanzada
    advanced_filled = advanced_impute_iterative(data2)
    diagnose_missing(advanced_filled, plot=False)  # sin nulos
    
    # 7) Guardado
    save_data(advanced_filled, "electric_vehicles_cleaned.csv")
